In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error

df=pd.read_csv('student_performance.csv')
print(f'Dataset loaded:{df.shape[0]} students,{df.shape[1]} columns')
print(f'Columns:{df.columns.tolist()}')
print(f'\nMissing values:{df.isnull().sum().sum()}')
print('\nFirst 5 rows:')
df.head()

Dataset loaded:30 students,13 columns
Columns:['student_id', 'name', 'age', 'gender', 'department', 'semester', 'math_score', 'science_score', 'english_score', 'programming_score', 'attendance_percentage', 'city', 'admission_year']

Missing values:0

First 5 rows:


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [ ]:
df_ml=df.copy()
le_gender=LabelEncoder()
df_ml['gender_encoded']=le_gender.fit_transform(df_ml['gender'])
print(f'Gender encoding:{dict(zip(le_gender.classes_,le_gender.transform(le_gender.classes_)))}')
le_dept=LabelEncoder()
df_ml['department_encoded']=le_dept.fit_transform(df_ml['department'])
print(f'department encoding:{dict(zip(le_dept.classes_,le_dept.transform(le_dept.classes_)))}')

df_ml[['gender','gender_encoded','department','department_encoded']].head(5)

Gender encoding:{'Female': np.int64(0), 'Male': np.int64(1)}
department encoding:{'Civil': np.int64(0), 'Computer Science': np.int64(1), 'Electronics': np.int64(2), 'Mechanical': np.int64(3)}


,gender,gender_encoded,department,department_encoded
0,Male,1,Computer Science,1
1,Female,0,Computer Science,1
2,Male,1,Electronics,2
3,Female,0,Mechanical,3
4,Male,1,Computer Science,1


In [ ]:
feature_cols=[
    'math_score',
    'science_score',
    'english_score',
    'attendance_percentage',
    'gender_encoded',
    'department_encoded'
]
X=df_ml[feature_cols]
y=df_ml['programming_score']




print(f'feature matrix x shape:{X.shape} students x features')
print(f'Target vector y shape:{y.shape} (one score per student)')
print(f'\nFeature columns:{feature_cols}')
print(f'Target column:programming_score')
print(f'\nTarget range:{y.min()} to {y.max()} (mean:{y.mean():.1f})')

feature matrix x shape:(30, 6) students x features
Target vector y shape:(30,) (one score per student)

Feature columns:['math_score', 'science_score', 'english_score', 'attendance_percentage', 'gender_encoded', 'department_encoded']
Target column:programming_score

Target range:38 to 97 (mean:67.6)


In [ ]:
x_train,x_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

print(f'Total students:{len(X)}')
print(f'Training students:{len(x_train)}({len(x_train)/len(X)*100:.0f}%)')
print(f'Testing Students:{len(x_test)}({len(x_test)/len(X)*100:.0f}%)')

print(f'\nTraining target range:{y_train.min()} to {y_train.max()}')
print(f'Testing target range : {y_test.min()}-{y_test.max()}')

Total students:30
Training students:24(80%)
Testing Students:6(20%)

Training target range:38 to 96
Testing target range : 39-97


In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print('Features scaled successfully!')
print(f'Training feature mean: {x_train_scaled.mean():.3f}')
print(f'Training feature standard deviation: {x_train_scaled.std():.3f}')

Features scaled successfully!
Training feature mean: -0.000
Training feature standard deviation: 1.000


In [ ]:
lr_model=LinearRegression()
lr_model.fit(x_train_scaled,y_train)
lr_pred=lr_model.predict(x_test_scaled)

lr_mae=mean_absolute_error(y_test,lr_pred)
lr_rmse=np.sqrt(mean_squared_error(y_test,lr_pred))
lr_r2=r2_score(y_test,lr_pred)

print(f'MAE :{lr_mae:.2f}(prediction are off by{lr_mae:.1f} ponits on average)')
print(f'RMSE :{lr_rmse:.2f} (error with penalty for large mistakes)')
print(f'R2:{lr_r2:.4f} ({lr_r2*100:.1f}% of programming score variation expalined)')
print()
print('Learned confficients (importance of each feature):')
for feat, coef in zip(feature_cols, lr_model.coef_):
    print(f'{feat:<28}: {coef:+.3f}')
print(f'{"bias (intercept)" :<28}: {lr_model.intercept_:.3f}')


MAE :9.37(prediction are off by9.4 ponits on average)
RMSE :11.47 (error with penalty for large mistakes)
R2:0.7356 (73.6% of programming score variation expalined)

Learned confficients (importance of each feature):
math_score                  : +20.256
science_score               : +7.612
english_score               : +2.396
attendance_percentage       : -12.886
gender_encoded              : -0.397
department_encoded          : -0.449
bias (intercept)            : 68.042


In [ ]:
dt_model=DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)
dt_model.fit(x_train_scaled,y_train)
dt_pred=dt_model.predict(x_test_scaled)

dt_mae=mean_absolute_error(y_test,dt_pred)
dt_rmse=np.sqrt(mean_squared_error(y_test,dt_pred))
dt_r2=r2_score(y_test,dt_pred)

print(f'MAE:{dt_mae:.2f}')
print(f'RMSE:{dt_rmse:.2f}')
print(f'R2:{dt_r2:.4f}')


MAE:10.75
RMSE:16.08
R2:0.4803


In [ ]:
dt_model=DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)
dt_model.fit(x_train_scaled,y_train)
dt_pred=dt_model.predict(x_test_scaled)

dt_mae=mean_absolute_error(y_test,dt_pred)
dt_rmse=np.sqrt(mean_squared_error(y_test,dt_pred))
dt_r2=r2_score(y_test,dt_pred)

print(f'MAE:{dt_mae:.2f}')
print(f'RMSE:{dt_rmse:.2f}')
print(f'R2:{dt_r2:.4f}')
